# [1] Imports & Motor-CAD 연결


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# [1] Imports & Motor-CAD 연결
# ─────────────────────────────────────────────────────────────────────────────
import pathlib
import sys
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat

# Repo root on path
repo_root = pathlib.Path.cwd().resolve()
while not ((repo_root / "tools").exists() or (repo_root / "tool").exists()) and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 패키지 reload (의존순서: model → plot → parse → facade)
import tools.motorCAD.pyMCAD.magnetic_model as _mm
import tools.motorCAD.pyMCAD.magnetic_plot as _mp
import tools.motorCAD.pyMCAD.magnetic_parse as _mparse
import tools.motorCAD.pyMCAD.magnetic as _mag
importlib.reload(_mm)
importlib.reload(_mp)
importlib.reload(_mparse)
importlib.reload(_mag)

from tools.motorCAD.pyMCAD import (
    get_magnetic_timeseries_from_file,
    mcad_default_export_dir,
    find_latest_mes,
    list_mes_files,
)
from tools.motorCAD.pyMCAD.magnetic_model import MagElement

import ansys.motorcad.core as pymotorcad

# Motor-CAD 연결
mcad = pymotorcad.MotorCAD(open_new_instance=False)
refMotFilePath=r"D:\KangDH\Thesis\e10\refModel\e10Turn6V261.mot"
HalfSCMotFilePath=r"D:\KangDH\Thesis\e10\SLFEA_Half\e10Turn6V261SLFEA_Half.mot"
SCMotFilePath=r"D:\KangDH\Thesis\e10\SLFEA\e10Turn6V261SLFEA.mot"

# mcad.load_from_file(refMotFilePath)
print("Motor-CAD connected")
# print(f"  MOT file: {mcad.get_variable('CurrentMotFilePath_MotorLAB')}")

Motor-CAD connected


# [2] e10 모터 파라미터 설정

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# [2] e10 모터 파라미터 설정
# ─────────────────────────────────────────────────────────────────────────────

# --- Conductor geometry (hairpin, rectangular) ---
# COND_WIDTH_MM = 2.5        # tangential width b [mm] (확인 필요 → Motor-CAD에서)
# COND_HEIGHT_MM = 2.5       # radial height h [mm] (확인 필요 → Motor-CAD에서)
SIGMA_CU = 5.8e7           # Cu conductivity @ 20°C [S/m]
# ACTIVE_LENGTH_MM = 100.0   # axial stack length [mm] (확인 필요)

# Motor-CAD에서 실제 값 읽기
try:
    COND_WIDTH_MM = float(mcad.get_variable("Copper_Width"))  # 슬롯 폭 / 병렬 수
    COND_HEIGHT_MM = float(mcad.get_variable("Copper_Height"))
    ACTIVE_LENGTH_MM = float(mcad.get_variable("Stator_Lam_Length"))
    n_parallel = int(mcad.get_variable("ParallelPaths"))
    n_turns = int(mcad.get_variable("MagTurnsConductor"))
    print(f"  Conductor: {COND_WIDTH_MM:.2f} x {COND_HEIGHT_MM:.2f} mm")
    print(f"  Active length: {ACTIVE_LENGTH_MM:.1f} mm")
    print(f"  Parallel paths: {n_parallel}, Turns/conductor: {n_turns}")
except Exception as e:
    print(f"  [WARN] Motor-CAD variable read failed: {e}")
    print(f"  Using default values: {COND_WIDTH_MM} x {COND_HEIGHT_MM} mm, L={ACTIVE_LENGTH_MM} mm")

# Convert to SI
b_m = COND_WIDTH_MM * 1e-3     # conductor tangential width [m]
h_m = COND_HEIGHT_MM * 1e-3    # conductor radial height [m]
L_a = ACTIVE_LENGTH_MM * 1e-3  # active length [m]

# --- Operating conditions ---
POLE_PAIRS = 4                 # 8-pole motor
SPEED_LIST = [2000, 4000, 16000]  # RPM

# Electrical frequency per speed
def speed_to_fe(speed_rpm, pole_pairs=POLE_PAIRS):
    return pole_pairs * speed_rpm / 60.0

print(f"\n  Speed → f_e: {[(s, f'{speed_to_fe(s):.0f} Hz') for s in SPEED_LIST]}")

# --- Skin depth & ξ table ---
MU_0 = 4 * np.pi * 1e-7
print(f"\n{'Speed [RPM]':>12} {'f_e [Hz]':>10} {'δ [mm]':>10} {'ξ = h/δ':>10}")
print("-" * 50)
for spd in SPEED_LIST:
    fe = speed_to_fe(spd)
    delta = 1.0 / np.sqrt(np.pi * fe * MU_0 * SIGMA_CU)
    xi_val = h_m / delta
    print(f"{spd:>12} {fe:>10.0f} {delta*1e3:>10.2f} {xi_val:>10.3f}")

  Conductor: 5.57 x 2.53 mm
  Active length: 150.0 mm
  Parallel paths: 1, Turns/conductor: 1

  Speed → f_e: [(2000, '133 Hz'), (4000, '267 Hz'), (16000, '1067 Hz')]

 Speed [RPM]   f_e [Hz]     δ [mm]    ξ = h/δ
--------------------------------------------------
        2000        133       5.72      0.442
        4000        267       4.05      0.625
       16000       1067       2.02      1.250


# [3] FEA 설정 + B-field TXT Export (속도별)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [3] FEA 설정 + B-field TXT Export (속도별)
# ─────────────────────────────────────────────────────────────────────────────
# 
# 옵션 A: Motor-CAD를 여기서 직접 실행
# 옵션 B: 이미 실행된 결과의 .mes를 로드하여 export만 수행
#
# 여기서는 옵션 A (실행 + export) 를 기본으로 합니다.
# 이미 결과가 있으면 DO_SOLVE=False로 설정하세요.

DO_SOLVE = True
PHASE_ADVANCE = 43.33
RMS_CURRENT = 460  # Ref model

# FEA export 설정
FIRST_STEP = 1
FINAL_STEP = mcad.get_variable("TorquePointsPerCycle")  # Motor-CAD 기본 TorquePointsPerCycle 정도
EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

out_root = Path(mcad_default_export_dir(mcad))
export_dir = out_root / "ACLossCalcExport"
export_dir.mkdir(parents=True, exist_ok=True)


# [3] 90-Point FEA Sweep & Directory Backup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [3] 180-Point FEA Sweep (Hybrid & FullFEA) & Directory Backup
# ─────────────────────────────────────────────────────────────────────────────
import os
import shutil
import json
from pathlib import Path
from datetime import datetime
from scipy.io import savemat

# 90-Point Sweep Definition (Total 180 points for both ProximityLossModels)
CURRENT_LIST = np.linspace(0.1, 460.0, 5)   # 5 currents
PHASE_LIST = np.linspace(0.0, 90.0, 6)      # 6 phase angles
PROXIMITY_MODELS = [1, 3]                  # 1: Hybrid, 3: FullFEA (TS)

# FEA export settings
FIRST_STEP = 1
EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

out_root = Path(mcad_default_export_dir(mcad))
backup_root = out_root / "ACLossCalcExport_Map"
backup_root.mkdir(parents=True, exist_ok=True)

sweep_results = []
total_points = len(SPEED_LIST) * len(CURRENT_LIST) * len(PHASE_LIST) * len(PROXIMITY_MODELS)
point_idx = 0

print(f"Starting 180-point sweep (speeds: {SPEED_LIST}, currents: {list(np.round(CURRENT_LIST, 1))}, phases: {list(np.round(PHASE_LIST, 1))})...")
print(f"Backup root: {backup_root}\n")

def calc_dc_loss_kw(resistance_ohm: float, rms_current_a: float) -> float:
    """3상 DC 손실 [kW] = 3 * R * I²."""
    return 3.0 * resistance_ohm * (rms_current_a ** 2) / 1000.0

# Get resistances once to calculate DC loss for FullFEA
try:
    R_total = float(mcad.get_variable("Resistance_MotorLAB")) * 4.0
    R_end = float(mcad.get_variable("EndWindingResistance_Lab")) * 4.0
    R_active = R_total - R_end
except Exception as e:
    R_total, R_end, R_active = 0.0, 0.0, 0.0
    print(f"  [WARN] Failed to read winding resistances: {e}")

for prox_model in PROXIMITY_MODELS:
    mcad.set_variable("ProximityLossModel", prox_model)
    mode_label = "Hybrid" if prox_model == 1 else "FullFEA"
    
    for speed in SPEED_LIST:
        mcad.set_variable("ShaftSpeed", speed)
        for current in CURRENT_LIST:
            mcad.set_variable("RMSCurrent", current)
            for phase in PHASE_LIST:
                mcad.set_variable("PhaseAdvance", phase)
                
                point_idx += 1
                print(f"[{point_idx}/{total_points}] [{mode_label}] Speed: {speed} RPM, Current: {current:.1f} A, Phase: {phase:.1f} deg")
                
                # 1. Run calculation
                print(f"  → Solving {mode_label} FEA...")
                mcad.do_magnetic_calculation()
                
                # Get TorquePointsPerCycle for TS or just use default
                torque_points = int(mcad.get_variable("TorquePointsPerCycle"))
                
                # 2. Get latest solved results directory
                try:
                    latest_mes = find_latest_mes(mcad)
                    active_results_dir = latest_mes.parent
                except Exception as e:
                    print(f"  [ERROR] Failed to locate latest .mes file: {e}")
                    continue
                
                # 3. Create destination folder
                point_folder_name = f"{mode_label}_Speed_{speed}RPM_{current:.1f}A_{phase:.1f}deg"
                dest_point_dir = backup_root / point_folder_name
                dest_results_dir = dest_point_dir / "FEResultsData"
                
                # 4. Copy active results folder
                print(f"  → Backing up results folder to: {point_folder_name}/FEResultsData")
                if dest_results_dir.exists():
                    shutil.rmtree(dest_results_dir)
                shutil.copytree(active_results_dir, dest_results_dir)
                
                # 5. Export B-field TXT file to the destination directory
                txt_path = dest_point_dir / "FEA_data.txt"
                print(f"  → Exporting B-field TXT to: {point_folder_name}/FEA_data.txt")
                mcad.save_fea_data(str(txt_path), FIRST_STEP, torque_points, EXPORT_COLUMNS, "", ",")
                
                # 6. Read losses and prepare point summary
                point_data = {
                    "proximity_model": prox_model,
                    "mode": mode_label,
                    "speed": speed,
                    "current": current,
                    "phase": phase,
                    "backup_dir": str(dest_point_dir)
                }
                
                if prox_model == 1:
                    # Hybrid scalar losses
                    try:
                        total_w = float(mcad.get_variable("ACLoss_Hybrid_Total"))
                        prox_w = float(mcad.get_variable("ACLoss_Hybrid_Prox_Total"))
                        skin_w = float(mcad.get_variable("ACLoss_Hybrid_SkinEffect_Total"))
                    except Exception as e:
                        total_w, prox_w, skin_w = 0.0, 0.0, 0.0
                        print(f"  [WARN] Failed to read hybrid losses: {e}")
                    point_data.update({
                        "hybrid_total_W": total_w,
                        "hybrid_prox_W": prox_w,
                        "hybrid_skin_W": skin_w,
                        "hybrid_total_kW": total_w / 1000.0,
                        "hybrid_prox_kW": prox_w / 1000.0,
                        "hybrid_skin_kW": skin_w / 1000.0,
                    })
                    print(f"  → Hybrid Loss: Total={total_w:.1f} W, Prox={prox_w:.1f} W, Skin={skin_w:.1f} W\n")
                else:
                    # FullFEA / TS scalar losses
                    try:
                        per_turn_str = mcad.get_variable("ACLoss_FEA_OnLoad_PerTurn")
                        if isinstance(per_turn_str, str):
                            per_turn_w = [float(x) for x in per_turn_str.split(":")]
                        else:
                            per_turn_w = list(per_turn_str)
                        per_turn_sum_kw = sum(per_turn_w) / 1000.0
                        total_kw = float(mcad.get_variable("ACLoss_FEA_OnLoad_Total")) / 1000.0
                    except Exception as e:
                        per_turn_w = []
                        per_turn_sum_kw, total_kw = 0.0, 0.0
                        print(f"  [WARN] Failed to read TS losses: {e}")
                    
                    dc_active_kw = calc_dc_loss_kw(R_active, current)
                    dc_end_kw = calc_dc_loss_kw(R_end, current)
                    ac_active_only_kw = per_turn_sum_kw - dc_active_kw
                    
                    point_data.update({
                        "ts_per_turn_W": per_turn_w,
                        "ts_per_turn_sum_kW": per_turn_sum_kw,
                        "ts_total_kW": total_kw,
                        "ts_dc_active_kW": dc_active_kw,
                        "ts_dc_end_kW": dc_end_kw,
                        "ts_ac_active_only_kW": ac_active_only_kw,
                    })
                    print(f"  → TS Loss: PerTurnSum={per_turn_sum_kw:.3f} kW, AC Active Only={ac_active_only_kw:.3f} kW, Total={total_kw:.3f} kW\n")
                
                sweep_results.append(point_data)

print(f"\n✓ Sweep complete! {point_idx} points processed and backed up under {backup_root}")

# ─────────────────────────────────────────────────────────────────────────────
# Save extracted scalar data to MAT and JSON
# ─────────────────────────────────────────────────────────────────────────────
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path("map_exports")
out_dir.mkdir(parents=True, exist_ok=True)
json_path = out_dir / f"JEET_ACLoss_180Map_Summary_{ts}.json"
mat_path = out_dir / f"JEET_ACLoss_180Map_Summary_{ts}.mat"

# Save JSON
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(sweep_results, f, ensure_ascii=False, indent=2)
print(f"Saved JSON summary: {json_path}")

# Save MAT (matlab compatible format)
hybrid_pts = [p for p in sweep_results if p["proximity_model"] == 1]
ts_pts = [p for p in sweep_results if p["proximity_model"] == 3]

def _arr(v):
    a = np.array(v, dtype=np.float64)
    return a.reshape(-1, 1)

mat_data = {
    "speeds_RPM": _arr([p["speed"] for p in hybrid_pts]),
    "currents_A": _arr([p["current"] for p in hybrid_pts]),
    "phases_deg": _arr([p["phase"] for p in hybrid_pts]),
    
    "hybrid_Total_kW": _arr([p["hybrid_total_kW"] for p in hybrid_pts]),
    "hybrid_Prox_kW": _arr([p["hybrid_prox_kW"] for p in hybrid_pts]),
    "hybrid_Skin_kW": _arr([p["hybrid_skin_kW"] for p in hybrid_pts]),
    
    "ts_OnLoad_PerTurnSum_kW": _arr([p["ts_per_turn_sum_kW"] for p in ts_pts]),
    "ts_Total_kW": _arr([p["ts_total_kW"] for p in ts_pts]),
    "ts_DC_Active_kW": _arr([p["ts_dc_active_kW"] for p in ts_pts]),
    "ts_DC_End_kW": _arr([p["ts_dc_end_kW"] for p in ts_pts]),
    "ts_ActiveOnly_kW": _arr([p["ts_ac_active_only_kW"] for p in ts_pts]),
}

if ts_pts and ts_pts[0].get("ts_per_turn_W"):
    mat_data["ts_OnLoad_PerTurn_kW"] = np.array([p["ts_per_turn_W"] for p in ts_pts], dtype=np.float64) / 1000.0

savemat(str(mat_path), mat_data, do_compression=True)
print(f"Saved MATLAB MAT: {mat_path}")


# [3c] 보완 스윕: 8000 RPM 추가 (4×4×4 맵 완성)

기존 [2000, 4000, 16000] RPM 데이터에 **8000 RPM**만 추가합니다.

| 항목 | 기존 | 추가 (8000 RPM) |
|---|---|---|
| 전류 | 5개 (0.1 ~ 460 A) | **4개** (near-zero 제외: 115, 230, 345, 460 A) |
| 위상 | 6개 (0 ~ 90°) | **4개** (0, 18, 54, 90°) |
| FEA 실행 | 180회 완료 | **+32회** (4×4×2모델) |

완료 후 JSON을 병합 저장하여 이후 셀에서 4속도 데이터로 사용합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [3c] 보완 스윕: 8000 RPM 추가
# 기존 JSON 로드 → 8000 RPM 실행 → 병합 저장
# ─────────────────────────────────────────────────────────────────────────────
import json, shutil, glob
import numpy as np
from pathlib import Path
from datetime import datetime
from tools.motorCAD.pyMCAD import calc_dc_loss_kw

ADDON_SPEED = 8000  # 추가할 속도 [RPM]

# 셀 [3]이 실행되지 않은 경우 기본값 정의
if 'CURRENT_LIST' not in globals():
    CURRENT_LIST = np.linspace(0.1, 460.0, 5)
if 'PHASE_LIST' not in globals():
    PHASE_LIST = np.linspace(0.0, 90.0, 6)
if 'PROXIMITY_MODELS' not in globals():
    PROXIMITY_MODELS = [1, 3]
if 'FIRST_STEP' not in globals():
    FIRST_STEP = 1
if 'EXPORT_COLUMNS' not in globals():
    EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"

# 기존 5전류 × 6위상 그리드에서 4×4 서브셋 선택
#   전류: near-zero(0.1 A) 제외 → 인덱스 [1,2,3,4]
#   위상: 균등 4개 → 인덱스 [0,1,3,5] = [0, 18, 54, 90] deg
ADDON_CURRENT_LIST = CURRENT_LIST[1:]          # [115.1, 230.1, 345.1, 460.0] A
ADDON_PHASE_LIST   = PHASE_LIST[[0, 1, 3, 5]]  # [0.0, 18.0, 54.0, 90.0] deg

print("=== 8000 RPM 보완 스윕 ===")
print(f"  전류: {np.round(ADDON_CURRENT_LIST, 1).tolist()} A  ({len(ADDON_CURRENT_LIST)}개)")
print(f"  위상: {ADDON_PHASE_LIST.tolist()} deg  ({len(ADDON_PHASE_LIST)}개)")
print(f"  모델: Hybrid(1) + FullFEA(3)")
total_addon = len(ADDON_CURRENT_LIST) * len(ADDON_PHASE_LIST) * len(PROXIMITY_MODELS)
print(f"  FEA 실행 예정: {total_addon}회\n")

# ── 기존 JSON 로드 ────────────────────────────────────────────────────────────
json_files = sorted(glob.glob("map_exports/JEET_ACLoss_180Map_Summary_*.json"))
if json_files:
    with open(json_files[-1], "r", encoding="utf-8") as f:
        sweep_results = json.load(f)
    print(f"기존 데이터 로드: {json_files[-1]}")
    print(f"  기존 포인트: {len(sweep_results)}개, "
          f"속도: {sorted(set(p['speed'] for p in sweep_results))} RPM\n")
elif 'sweep_results' not in globals():
    raise RuntimeError("기존 sweep 데이터 없음. 먼저 셀 [3b]를 실행하세요.")

# ── 저항값 읽기 (셀 [3] 미실행 시 여기서 직접 읽음) ─────────────────────────
if 'R_active' not in globals() or 'R_end' not in globals():
    try:
        _R_total = float(mcad.get_variable("Resistance_MotorLAB")) * 4.0
        R_end    = float(mcad.get_variable("EndWindingResistance_Lab")) * 4.0
        R_active = _R_total - R_end
        print(f"저항값 읽기 완료: R_active={R_active:.6f} Ω, R_end={R_end:.6f} Ω\n")
    except Exception as e:
        R_active = R_end = 0.0
        print(f"  [WARN] 저항값 읽기 실패: {e} → DC 손실 = 0으로 처리\n")

# ── 중복 방지: 이미 8000 RPM 있으면 스킵 ─────────────────────────────────────
existing_8k = [p for p in sweep_results if p["speed"] == ADDON_SPEED]
if existing_8k:
    print(f"이미 {ADDON_SPEED} RPM 데이터 {len(existing_8k)}개 존재 → 스킵")
else:
    out_root    = Path(mcad_default_export_dir(mcad))
    backup_root = out_root / "ACLossCalcExport_Map"
    backup_root.mkdir(parents=True, exist_ok=True)

    pt_idx = 0
    mcad.set_variable("ShaftSpeed", ADDON_SPEED)

    for prox_model in PROXIMITY_MODELS:
        mcad.set_variable("ProximityLossModel", prox_model)
        mode_label = "Hybrid" if prox_model == 1 else "FullFEA"

        for current in ADDON_CURRENT_LIST:
            mcad.set_variable("RMSCurrent", current)
            for phase in ADDON_PHASE_LIST:
                mcad.set_variable("PhaseAdvance", phase)
                pt_idx += 1
                print(f"[{pt_idx}/{total_addon}] [{mode_label}] "
                      f"{ADDON_SPEED} RPM, {current:.1f} A, {phase:.1f}°")

                mcad.do_magnetic_calculation()
                torque_points = int(mcad.get_variable("TorquePointsPerCycle"))

                try:
                    latest_mes      = find_latest_mes(mcad) # 최근 .mes 파일 경로
                    active_res_dir  = latest_mes.parent # FEA 결과 폴더 경로
                except Exception as e:
                    print(f"  [ERROR] {e}")
                    continue

                folder  = f"{mode_label}_Speed_{ADDON_SPEED}RPM_{current:.1f}A_{phase:.1f}deg" # FEA 결과 폴더 이름
                dst_fe  = backup_root / folder / "FEResultsData" # FEA 결과 폴더 경로
                if dst_fe.exists(): shutil.rmtree(dst_fe) # 기존 FEA 결과 삭제
                shutil.copytree(active_res_dir, dst_fe) # FEA 결과 폴더 복사

                txt_path = backup_root / folder / "FEA_data.txt" # FEA 데이터 파일 경로
                mcad.save_fea_data(str(txt_path), FIRST_STEP, torque_points, EXPORT_COLUMNS, "", ",")
                
                # FEA 데이터 포인트 생성
                pt = {"proximity_model": prox_model, "mode": mode_label,
                      "speed": ADDON_SPEED, "current": current, "phase": phase,
                      "backup_dir": str(backup_root / folder)}

                if prox_model == 1:
                    try:
                        tw = float(mcad.get_variable("ACLoss_Hybrid_Total"))
                        pw = float(mcad.get_variable("ACLoss_Hybrid_Prox_Total"))
                        sw = float(mcad.get_variable("ACLoss_Hybrid_SkinEffect_Total"))
                    except:
                        tw = pw = sw = 0.0
                    pt.update({
                        "hybrid_total_W": tw, "hybrid_prox_W": pw, "hybrid_skin_W": sw,
                        "hybrid_total_kW": tw/1e3, "hybrid_prox_kW": pw/1e3,
                        "hybrid_skin_kW": sw/1e3,
                    })
                    print(f"  → Hybrid Total: {tw:.1f} W\n")
                else:
                    try:
                        pt_str = mcad.get_variable("ACLoss_FEA_OnLoad_PerTurn")
                        ptw    = [float(x) for x in pt_str.split(":")] \
                                 if isinstance(pt_str, str) else list(pt_str)
                        pts_kw = sum(ptw) / 1e3
                        tot_kw = float(mcad.get_variable("ACLoss_FEA_OnLoad_Total")) / 1e3
                    except:
                        ptw, pts_kw, tot_kw = [], 0., 0.
                    dc_act = calc_dc_loss_kw(R_active, current)
                    dc_end = calc_dc_loss_kw(R_end, current)
                    ac_act = pts_kw - dc_act
                    pt.update({
                        "ts_per_turn_W": ptw, "ts_per_turn_sum_kW": pts_kw,
                        "ts_total_kW": tot_kw, "ts_dc_active_kW": dc_act,
                        "ts_dc_end_kW": dc_end, "ts_ac_active_only_kW": ac_act,
                    })
                    print(f"  → FullFEA AC Active Only: {ac_act:.3f} kW\n")

                sweep_results.append(pt)

    # ── 병합 저장 ─────────────────────────────────────────────────────────────
    ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = Path("map_exports")
    save_path = out_dir / f"JEET_ACLoss_4Speed_Map_Summary_{ts}.json"
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(sweep_results, f, ensure_ascii=False, indent=2)

    spds = sorted(set(p["speed"] for p in sweep_results))
    print(f"\n✓ 병합 완료: 총 {len(sweep_results)}포인트 | 속도: {spds} RPM")
    print(f"✓ 저장: {save_path}")

# ── 스윕 구성 확인 ────────────────────────────────────────────────────────────
print("\n=== 최종 데이터 구성 ===")
for spd in sorted(set(p["speed"] for p in sweep_results)):
    h_pts = [p for p in sweep_results if p["speed"]==spd and p["proximity_model"]==1]
    f_pts = [p for p in sweep_results if p["speed"]==spd and p["proximity_model"]==3]
    print(f"  {spd:5d} RPM → Hybrid: {len(h_pts):2d}pt, FullFEA: {len(f_pts):2d}pt")

=== 8000 RPM 보완 스윕 ===
  전류: [115.1, 230.0, 345.0, 460.0] A  (4개)
  위상: [0.0, 18.0, 54.0, 90.0] deg  (4개)
  모델: Hybrid(1) + FullFEA(3)
  FEA 실행 예정: 32회

기존 데이터 로드: map_exports\JEET_ACLoss_180Map_Summary_20260620_055628.json
  기존 포인트: 180개, 속도: [2000, 4000, 16000] RPM

저항값 읽기 완료: R_active=0.087978 Ω, R_end=0.071417 Ω

[1/32] [Hybrid] 8000 RPM, 115.1 A, 0.0°


# [4] id-iq 평면 AC Active Only 손실 Surface 플롯 (속도별)

ProximityLossModel = 1(Hybrid) 및 3(FullFEA/TS) 각각에 대해 속도별로 $I_d, I_q$ 평면에서의 AC Active Only 손실 Surface 플롯을 시각화합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [4] id-iq 평면 AC Active Only 손실 Surface 플롯 (Hybrid vs FullFEA 대화형 비교)
# ─────────────────────────────────────────────────────────────────────────────
# [Matplotlib 백엔드 설정]
# - 'auto'     : 환경 자동 감지 (VS Code -> widget, 브라우저 -> inline)
# - 'inline'   : 정적 이미지 출력 (웹 브라우저 JupyterLab에서 렌더링 에러 발생 시 이 값으로 설정하세요!)
# - 'widget'   : VS Code 및 JupyterLab용 대화형 플롯 (ipympl 필요)
# - 'notebook' : 클래식 Jupyter Notebook용 대화형 플롯 (nbagg)
PLOT_BACKEND = 'auto'  # <-- 브라우저에서 플롯이 안 뜨거나 렌더링 에러가 나면 'inline'으로 변경하고 다시 실행하세요!

import os
import sys


try:
    import IPython
    shell = IPython.get_ipython()
    if shell is not None:
        has_vscode_env = any(k.startswith('VSCODE_') for k in os.environ.keys())
        has_vscode_modules = any('vscode' in m.lower() for m in sys.modules.keys())
        
        # 1. 자동 감지 시 백엔드 결정
        selected_backend = PLOT_BACKEND
        if selected_backend == 'auto':
            if has_vscode_env and has_vscode_modules:
                selected_backend = 'widget'
            else:
                selected_backend = 'inline'
        
        print("--- Matplotlib Backend Config ---")
        print(f"  [Environment] VS Code Env={has_vscode_env}, VS Code Modules={has_vscode_modules}")
        print(f"  [Selection] Configured Mode='{PLOT_BACKEND}' -> Selected Backend='{selected_backend}'")
        
        # 2. 백엔드 적용
        if selected_backend == 'widget':
            shell.run_line_magic('matplotlib', 'widget')
            print("  -> Interactive backend 'widget' (ipympl) enabled.")
            print("  -> [Tip] 만약 웹 브라우저에서 'model not found' 등의 렌더링 에러가 발생하거나")
            print("           그래프가 표시되지 않는다면, 셀 맨 위의 PLOT_BACKEND = 'inline' 으로 변경 후 다시 실행하세요.")
        elif selected_backend == 'notebook':
            shell.run_line_magic('matplotlib', 'notebook')
            print("  -> Interactive backend 'notebook' (nbagg) enabled.")
        elif selected_backend == 'inline':
            shell.run_line_magic('matplotlib', 'inline')
            print("  -> Static backend 'inline' enabled (non-interactive).")
        else:
            shell.run_line_magic('matplotlib', 'inline')
            print(f"  -> Unknown backend '{selected_backend}', fallback to 'inline' enabled.")
except Exception as e:
    print(f"Failed to set matplotlib backend: {e}")

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as mpatches

# 180포인트 결과가 sweep_results에 없으면 최신 JSON 파일 로드 시도
if 'sweep_results' not in globals() or not sweep_results:
    import glob
    from pathlib import Path
    import json
    
    json_files = sorted(glob.glob("map_exports/JEET_ACLoss_180Map_Summary_*.json"))
    if json_files:
        latest_json = json_files[-1]
        print(f"Loading latest sweep results from: {latest_json}")
        with open(latest_json, "r", encoding="utf-8") as f:
            sweep_results = json.load(f)
    else:
        print("Error: sweep_results not found in memory or map_exports!")
        sweep_results = []

if sweep_results:
    # 1. 데이터 분리 및 id, iq 변환
    hybrid_data = [p for p in sweep_results if p["proximity_model"] == 1]
    ts_data = [p for p in sweep_results if p["proximity_model"] == 3]
    
    def process_pts(pts, is_hybrid):
        speeds = np.array([p["speed"] for p in pts])
        currents = np.array([p["current"] for p in pts])
        phases = np.array([p["phase"] for p in pts])
        
        # dq 변환
        amplitude = currents * np.sqrt(2)
        phase_rad = (phases + 90) * np.pi / 180.0
        id_vals = amplitude * np.cos(phase_rad)
        iq_vals = amplitude * np.sin(phase_rad)
        
        if is_hybrid:
            losses = np.array([p["hybrid_total_kW"] for p in pts])
        else:
            losses = np.array([p["ts_ac_active_only_kW"] for p in pts])
            
        return speeds, id_vals, iq_vals, losses, pts

    # 속도별 뚜렷한 색상 설정 (2000: Cyan, 4000: LimeGreen, 16000: Tomato)
    speed_colors = {
        2000: 'cyan',
        4000: 'limegreen',
        8000: 'orange',
        16000: 'tomato'
    }
    default_colors = ['cyan', 'limegreen', 'tomato']
    
    def create_interactive_comparison_plot(pts_hybrid, pts_ts):
        speeds_h, id_h, iq_h, losses_h, raw_h = process_pts(pts_hybrid, is_hybrid=True)
        speeds_f, id_f, iq_f, losses_f, raw_f = process_pts(pts_ts, is_hybrid=False)
        
        currents_h = np.array([p["current"] for p in raw_h])
        phases_h = np.array([p["phase"] for p in raw_h])
        currents_f = np.array([p["current"] for p in raw_f])
        phases_f = np.array([p['phase'] for p in raw_f])
        
        unique_speeds = sorted(list(set(speeds_h)))
        
        # 3개 서브플롯 구성 (1행 3열: Hybrid 3D | FullFEA 3D | Comparison 2D)
        fig = plt.figure(figsize=(18, 5.5))
        fig.suptitle("AC Loss Comparison Map: Hybrid vs FullFEA (HalfSC Model)", fontsize=13, fontweight='bold')
        
        # Subplot 1: Hybrid 3D Map
        ax_left = fig.add_subplot(131, projection='3d')
        ax_left.set_title("Hybrid (ProximityLossModel = 1)", fontsize=11, fontweight='bold')
        
        # Subplot 2: FullFEA 3D Map
        ax_mid = fig.add_subplot(132, projection='3d')
        ax_mid.set_title("FullFEA (ProximityLossModel = 3)", fontsize=11, fontweight='bold')
        
        # 3D Surfaces 그리기
        legend_patches_h = []
        legend_patches_f = []
        
        for i, spd in enumerate(unique_speeds):
            idx_h = (speeds_h == spd)
            if np.any(idx_h):
                color = speed_colors.get(spd, default_colors[i % len(default_colors)])
                ax_left.plot_trisurf(id_h[idx_h], iq_h[idx_h], losses_h[idx_h], color=color, edgecolor='none', alpha=0.35)
                legend_patches_h.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
                
            idx_f = (speeds_f == spd)
            if np.any(idx_f):
                color = speed_colors.get(spd, default_colors[i % len(default_colors)])
                ax_mid.plot_trisurf(id_f[idx_f], iq_f[idx_f], losses_f[idx_f], color=color, edgecolor='none', alpha=0.35)
                legend_patches_f.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
                
        # 클릭용 Scatter 점 레이어
        sc_h = ax_left.scatter(id_h, iq_h, losses_h, c='grey', s=25, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
        sc_f = ax_mid.scatter(id_f, iq_f, losses_f, c='grey', s=25, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
        
        for ax, lp in [(ax_left, legend_patches_h), (ax_mid, legend_patches_f)]:
            ax.set_xlabel("I_d [A]", fontsize=8, labelpad=7)
            ax.set_ylabel("I_q [A]", fontsize=8, labelpad=7)
            ax.set_zlabel("AC Loss [kW]", fontsize=8, labelpad=7)
            ax.legend(handles=lp, fontsize=8, loc="upper right")
            
        # Subplot 3: 2D Comparison Curve
        ax_right = fig.add_subplot(133)
        ax_right.text(0.5, 0.5, "Click any point in left/middle 3D plots\nand press 'Space' to draw comparison curves", 
                     ha="center", va="center", fontsize=10, color="gray")
        ax_right.set_xlabel("Speed [RPM]", fontsize=9)
        ax_right.set_ylabel("AC Loss [kW]", fontsize=9)
        ax_right.grid(True, linestyle="--", alpha=0.5)
        
        # 공유 선택 상태 및 하이라이트 상태
        selected_pt = {"current": None, "phase": None, "id": None, "iq": None}
        highlights_h = []
        highlights_f = []
        
        # 양쪽 3D 플롯에 텍스트 안내 표시
        annotation_h = ax_left.text2D(0.02, 0.95, "", transform=ax_left.transAxes, 
                                      bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
        annotation_f = ax_mid.text2D(0.02, 0.95, "", transform=ax_mid.transAxes, 
                                     bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
        annotation_h.set_visible(False)
        annotation_f.set_visible(False)
        
        def on_pick(event):
            if event.artist not in [sc_h, sc_f]:
                return
            
            idx = event.ind[0]
            if event.artist == sc_h:
                curr = raw_h[idx]["current"]
                ph = raw_h[idx]["phase"]
            else:
                curr = raw_f[idx]["current"]
                ph = raw_f[idx]["phase"]
                
            selected_pt["current"] = curr
            selected_pt["phase"] = ph
            
            amp = curr * np.sqrt(2)
            phase_rad = (ph + 90) * np.pi / 180.0
            selected_pt["id"] = amp * np.cos(phase_rad)
            selected_pt["iq"] = amp * np.sin(phase_rad)
            
            # 이전 하이라이트 지우기
            for h in highlights_h + highlights_f:
                h.remove()
            highlights_h.clear()
            highlights_f.clear()
            
            # 양쪽 3D 플롯 모두 하이라이트 추가 (동일한 current/phase)
            same_h_idx = (currents_h == curr) & (phases_h == ph)
            hh = ax_left.scatter(id_h[same_h_idx], iq_h[same_h_idx], losses_h[same_h_idx], 
                                 color='red', s=70, edgecolors='black', linewidths=1.8, zorder=10)
            highlights_h.append(hh)
            
            same_f_idx = (currents_f == curr) & (phases_f == ph)
            hf = ax_mid.scatter(id_f[same_f_idx], iq_f[same_f_idx], losses_f[same_f_idx], 
                                color='red', s=70, edgecolors='black', linewidths=1.8, zorder=10)
            highlights_f.append(hf)
            
            # 텍스트 안내 박스 업데이트
            msg = (
                f"Selected Point:\n"
                f"I_rms: {curr:.1f} A, Phase: {ph:.1f} deg\n"
                f"Id: {selected_pt['id']:.1f} A, Iq: {selected_pt['iq']:.1f} A\n"
                f"→ Press 'Space' to compare"
            )
            for annot in [annotation_h, annotation_f]:
                annot.set_text(msg)
                annot.set_visible(True)
                
            fig.canvas.draw_idle()
            
        def on_key(event):
            if event.key != ' ':
                return
            if selected_pt["current"] is None:
                return
            
            ax_right.clear()
            
            curr = selected_pt["current"]
            ph = selected_pt["phase"]
            
            curve_speeds = []
            curve_losses_h = []
            curve_losses_f = []
            
            # Hybrid와 FullFEA 결과 비교 매칭
            for spd in unique_speeds:
                match_h = [p for p in raw_h if p["speed"] == spd and np.isclose(p["current"], curr) and np.isclose(p["phase"], ph)]
                match_f = [p for p in raw_f if p["speed"] == spd and np.isclose(p["current"], curr) and np.isclose(p["phase"], ph)]
                if match_h and match_f:
                    curve_speeds.append(spd)
                    curve_losses_h.append(match_h[0]["hybrid_total_kW"])
                    curve_losses_f.append(match_f[0]["ts_ac_active_only_kW"])
            
            # 두 곡선 겹쳐 그리기
            ax_right.plot(curve_speeds, curve_losses_h, marker='o', linestyle='-', color='dodgerblue', linewidth=2, label="Hybrid AC Total")
            ax_right.plot(curve_speeds, curve_losses_f, marker='*', linestyle='--', color='crimson', linewidth=2, label="FullFEA AC Active Only")
            
            # 수치 텍스트 표시
            for xs, yh, yf in zip(curve_speeds, curve_losses_h, curve_losses_f):
                ax_right.annotate(f"{yh:.2f}", xy=(xs, yh), xytext=(4, 4), textcoords="offset points", fontsize=8, color="dodgerblue")
                ax_right.annotate(f"{yf:.2f}", xy=(xs, yf), xytext=(4, -12), textcoords="offset points", fontsize=8, color="crimson")
                
            ax_right.set_title(f"AC Loss vs Speed Comparison\n(I_rms={curr:.1f} A, Phase={ph:.1f} deg)", fontsize=11, fontweight='bold')
            ax_right.set_xlabel("Speed [RPM]", fontsize=9)
            ax_right.set_ylabel("AC Loss [kW]", fontsize=9)
            ax_right.grid(True, linestyle="--", alpha=0.5)
            ax_right.legend(fontsize=9, loc="upper left")
            
            fig.canvas.draw_idle()
            
        fig.canvas.mpl_connect('pick_event', on_pick)
        fig.canvas.mpl_connect('key_press_event', on_key)
        plt.tight_layout()
        plt.show()
        
    if len(hybrid_data) > 0 and len(ts_data) > 0:
        create_interactive_comparison_plot(hybrid_data, ts_data)
    else:
        print("Error: Need both Hybrid and FullFEA data in sweep results to build comparison plot.")
else:
    print("No sweep results found to plot.")

# [5] Adjustment Factor (AF) 맵 계산

`AF = FullFEA_AC_active_only / Hybrid_AC_total` 을 각 운전점 **(speed, id, iq)** 별로 계산합니다.

**AF가 운전점마다 달라지는 물리적 이유:**
- **Skin effect** → 주파수(속도)에만 의존: AF↑ as speed↑
- **Proximity effect** → 슬롯 내 누설 플럭스 B² ∝ Iq²에 비례: Iq↑이면 Hybrid 오차 패턴 변화
- **Id**는 상대적으로 AF에 영향 적음 (d축은 자속 약화, eddy 유발 플럭스와 직접 연관 낮음)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [5] Adjustment Factor (AF) Map: FullFEA / Hybrid 비율 계산
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import json
import glob
from pathlib import Path

# sweep_results가 없으면 최신 JSON 로드
if 'sweep_results' not in globals() or not sweep_results:
    json_files = sorted(glob.glob("map_exports/JEET_ACLoss_180Map_Summary_*.json"))
    if json_files:
        with open(json_files[-1], "r", encoding="utf-8") as f:
            sweep_results = json.load(f)
    else:
        raise RuntimeError("sweep_results not found. Run cell [3] first.")

hybrid_data = [p for p in sweep_results if p["proximity_model"] == 1]
ts_data     = [p for p in sweep_results if p["proximity_model"] == 3]

# (speed, current, phase) 기준 매칭 → AF 계산
af_points = []
for ts_pt in ts_data:
    spd  = ts_pt["speed"]
    curr = ts_pt["current"]
    ph   = ts_pt["phase"]

    matches = [p for p in hybrid_data
               if p["speed"] == spd
               and np.isclose(p["current"], curr, atol=1e-3)
               and np.isclose(p["phase"],   ph,   atol=1e-3)]
    if not matches:
        continue
    h_pt = matches[0]

    h_ac = h_pt["hybrid_total_kW"]
    f_ac = ts_pt["ts_ac_active_only_kW"]

    if h_ac < 1e-4:          # 전류 거의 0 → skip (분모 불안정)
        continue

    af = f_ac / h_ac

    # dq 변환 (peak)
    amp    = curr * np.sqrt(2)
    ph_rad = (ph + 90.0) * np.pi / 180.0
    id_a   = amp * np.cos(ph_rad)
    iq_a   = amp * np.sin(ph_rad)

    af_points.append({
        "speed_rpm":   spd,
        "speed_kRPM":  spd / 1000.0,
        "current_rms": curr,
        "phase_deg":   ph,
        "id_A":        id_a,
        "iq_A":        iq_a,
        "hybrid_ac_kW": h_ac,
        "fea_ac_kW":    f_ac,
        "AF":           af,
    })

print(f"AF 계산 완료: {len(af_points)}개 운전점\n")
print(f"{'Speed[kRPM]':>12} {'Id[A]':>9} {'Iq[A]':>9} {'H_AC[kW]':>11} {'F_AC[kW]':>11} {'AF[-]':>7}")
print("─" * 65)
for p in af_points:
    print(f"{p['speed_kRPM']:>12.1f} {p['id_A']:>9.1f} {p['iq_A']:>9.1f} "
          f"{p['hybrid_ac_kW']:>11.3f} {p['fea_ac_kW']:>11.3f} {p['AF']:>7.3f}")

af_arr = np.array([p["AF"] for p in af_points])
print(f"\nAF 통계: min={af_arr.min():.3f}, max={af_arr.max():.3f}, "
      f"mean={af_arr.mean():.3f}, std={af_arr.std():.3f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [6] Polynomial Regression: AF(speed, id, iq) 다항식 피팅
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

speeds_k  = np.array([p["speed_kRPM"]  for p in af_points])
id_arr    = np.array([p["id_A"]         for p in af_points])
iq_arr    = np.array([p["iq_A"]         for p in af_points])
af_arr    = np.array([p["AF"]           for p in af_points])
curr_arr  = np.array([p["current_rms"]  for p in af_points])

# ── 방법 A: 최대 전류에서 속도만의 2차 다항식 (Motor-CAD Lab 수식용) ─────────
max_curr      = curr_arr.max()
mask_maxcurr  = np.isclose(curr_arr, max_curr, rtol=0.01)
spd_mc, af_mc = speeds_k[mask_maxcurr], af_arr[mask_maxcurr]

# 오름차순 정렬
sort_idx = np.argsort(spd_mc)
spd_mc, af_mc = spd_mc[sort_idx], af_mc[sort_idx]

# AF(s) = a2·s² + a1·s + a0  (s = speed in kRPM)
coeffs_A  = np.polyfit(spd_mc, af_mc, deg=2)   # [a2, a1, a0]
af_A_fit  = np.polyval(coeffs_A, spd_mc)
a2, a1, a0 = coeffs_A

print("=== 방법 A: 속도만의 2차 다항식 (최대 전류 기준) ===")
print(f"  I_rms = {max_curr:.1f} A 기준")
print(f"  AF(s) = {a2:.6f}·s² + {a1:.6f}·s + {a0:.6f}   (s: kRPM)\n")
print(f"  피팅 확인:")
for s, ref, fit in zip(spd_mc, af_mc, af_A_fit):
    print(f"    {s:.0f} kRPM: AF_ref={ref:.3f}, AF_fit={fit:.3f}, Δ={fit-ref:+.3f}")

# Motor-CAD Lab 수식 문자열 생성
# Extra loss = (AF - 1) × Hybrid_AC  →  추가 손실로 등록
lab_formula_extra = (
    f"(({a2:.6f}*(Speed/1000)^2 + {a1:.6f}*(Speed/1000) + {a0:.6f}) - 1)"
    f" * Stator_Copper_Loss_AC"
)
print(f"\n  [Motor-CAD Lab 수식 — 'Extra AC loss']")
print(f"  {lab_formula_extra}")

# ── 방법 B: (speed_kRPM, id, iq) 2차 다항식 (Python 보정 함수용) ────────────
# Feature: [1, s, id, iq, s², id², iq², s·id, s·iq, id·iq]
_feat_labels = ["1", "s", "id", "iq", "s²", "id²", "iq²", "s·id", "s·iq", "id·iq"]

def _build_features(s, id_v, iq_v):
    s, id_v, iq_v = np.asarray(s, float), np.asarray(id_v, float), np.asarray(iq_v, float)
    return np.column_stack([
        np.ones_like(s),
        s, id_v, iq_v,
        s**2, id_v**2, iq_v**2,
        s * id_v, s * iq_v, id_v * iq_v,
    ])

X         = _build_features(speeds_k, id_arr, iq_arr)
coeffs_B, _, _, _ = np.linalg.lstsq(X, af_arr, rcond=None)
af_B_fit  = X @ coeffs_B
rmse_B    = np.sqrt(np.mean((af_arr - af_B_fit)**2))
r2_B      = 1.0 - np.sum((af_arr - af_B_fit)**2) / np.sum((af_arr - af_arr.mean())**2)

print(f"\n=== 방법 B: (speed_kRPM, id, iq) 2차 다항식 ===")
print(f"  RMSE = {rmse_B:.4f},  R² = {r2_B:.4f}")
print(f"  계수:")
for lab, c in zip(_feat_labels, coeffs_B):
    print(f"    {lab:>8s}: {c:+.6f}")

# ── 보정 함수 정의 (이후 셀에서 재사용) ──────────────────────────────────────
_coeff_A = coeffs_A.copy()
_coeff_B = coeffs_B.copy()

def af_from_poly3d(speed_rpm, id_peak_a, iq_peak_a):
    """AF = FullFEA_AC / Hybrid_AC 추정 (speed, id, iq 3D 2차 다항식).

    Args:
        speed_rpm  : 회전속도 [RPM]  (scalar 또는 ndarray)
        id_peak_a  : d축 전류 [A, peak]
        iq_peak_a  : q축 전류 [A, peak]
    Returns:
        AF [-] — Hybrid AC 손실에 곱할 보정계수
    """
    s   = np.asarray(speed_rpm,  float) / 1000.0
    idv = np.asarray(id_peak_a,  float)
    iqv = np.asarray(iq_peak_a,  float)
    feats = _build_features(s.ravel(), idv.ravel(), iqv.ravel())
    af    = feats @ _coeff_B
    return af.reshape(np.asarray(speed_rpm).shape) if np.ndim(speed_rpm) > 0 else float(af[0])

def af_from_speed_only(speed_rpm):
    """AF 추정 (속도만의 2차 다항식, I=Imax 기준). Motor-CAD Lab 수식 검증용."""
    return np.polyval(_coeff_A, np.asarray(speed_rpm, float) / 1000.0)

print(f"\n  ✓ 보정 함수 준비 완료:")
print(f"    af_from_poly3d(speed_rpm, id_peak_a, iq_peak_a)  → 방법 B")
print(f"    af_from_speed_only(speed_rpm)                     → 방법 A")

# [6.5] AF vs Speed 곡선 — 운전점(전류 × 위상각)별

튜토리얼 그래프와 동일한 형식: 각 **(I_rms, 위상각)** 조합에서의 AF vs 속도 곡선을 표시합니다.  
색상 = 전류 크기, 선스타일 = 위상각.  방법-A 다항식 피팅선(검정 굵은 점선)도 함께 표시합니다.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [6.5] AF vs Speed 곡선: 운전점별 (전류 × 위상각)
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

# 고유 운전점 목록
unique_currents_s = sorted(set(round(p["current_rms"], 0) for p in af_points))
unique_phases_s   = sorted(set(round(p["phase_deg"],   0) for p in af_points))
unique_speeds_s   = sorted(set(p["speed_rpm"] for p in af_points))

n_curr_s  = len(unique_currents_s)
n_phase_s = len(unique_phases_s)

# 색상: 전류별 (plasma), 선스타일: 위상각별
colors_s  = [plt.cm.plasma(i / max(1, n_curr_s - 1)) for i in range(n_curr_s)]
lstyles_s = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 5))]

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_title("Adjustment Factor  AF = FullFEA_AC / Hybrid_AC  vs Speed (운전점별)",
             fontsize=12, fontweight='bold')

for ki, curr in enumerate(unique_currents_s):
    for li, ph in enumerate(unique_phases_s):
        pts = sorted(
            [p for p in af_points
             if np.isclose(p["current_rms"], curr, atol=0.6)
             and np.isclose(p["phase_deg"],  ph,   atol=0.6)],
            key=lambda x: x["speed_rpm"]
        )
        if len(pts) < 2:
            continue
        spds = [p["speed_rpm"] for p in pts]
        afs  = [p["AF"]        for p in pts]
        ax.plot(spds, afs,
                marker='o', markersize=5,
                linestyle=lstyles_s[li % len(lstyles_s)],
                color=colors_s[ki], linewidth=1.5,
                label=f"I={curr:.0f} A, φ={ph:.0f}°")

# 방법 A 다항식 피팅선 (최대 전류 기준, 검정 굵은 점선)
spd_fit  = np.linspace(min(unique_speeds_s) * 0.9, max(unique_speeds_s) * 1.05, 300)
af_A_fit = af_from_speed_only(spd_fit)
a2, a1, a0 = _coeff_A
eq_str = f"y = {a2:.4f}·x² {a1:+.4f}·x {a0:+.4f}  (x: kRPM)"
ax.plot(spd_fit, af_A_fit, 'k--', linewidth=2.5,
        label=f"Poly-A fit (I_max={max_curr:.0f} A)")
ax.text(0.97, 0.97, eq_str, transform=ax.transAxes, fontsize=9,
        va='top', ha='right', bbox=dict(boxstyle='round', fc='white', alpha=0.85))

ax.axhline(y=1.0, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label="AF = 1")
ax.set_xlabel("Speed [RPM]", fontsize=11)
ax.set_ylabel("Adjustment factor [-]", fontsize=11)
ax.legend(fontsize=7.5, loc='upper right', ncol=2, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("map_exports/AF_vs_speed_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("저장: map_exports/AF_vs_speed_curves.png")

# [7] AF 맵 시각화

- **좌측 패널(속도별)**: id-iq 평면에서의 AF 산점도 + 3D 다항식 등고선 오버레이
- **우측 패널**: 전류별 AF vs 속도 곡선 + 방법 A 피팅선 비교

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [7] AF 맵 시각화: id-iq 평면 (속도별) + AF vs Speed (전류별)
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

unique_speeds_v = sorted(set(p["speed_rpm"] for p in af_points))
n_spd_v = len(unique_speeds_v)

fig, axes = plt.subplots(1, n_spd_v + 1, figsize=(5.2 * (n_spd_v + 1), 4.8))
if n_spd_v + 1 == 1:
    axes = [axes]
fig.suptitle("Adjustment Factor  AF = FullFEA_AC / Hybrid_AC", fontsize=13, fontweight='bold')

af_vals_all = np.array([p["AF"] for p in af_points])
vmin_af = max(0.5, af_vals_all.min() - 0.1)
vmax_af = af_vals_all.max() + 0.1

# ── 속도별 id-iq 평면 산점도 + 다항식 등고선 ──────────────────────────────
for ax, spd in zip(axes[:n_spd_v], unique_speeds_v):
    pts  = [p for p in af_points if p["speed_rpm"] == spd]
    id_v = np.array([p["id_A"] for p in pts])
    iq_v = np.array([p["iq_A"] for p in pts])
    af_v = np.array([p["AF"]   for p in pts])

    sc = ax.scatter(id_v, iq_v, c=af_v, cmap='plasma', s=90,
                    edgecolors='k', linewidths=0.6,
                    vmin=vmin_af, vmax=vmax_af, zorder=3)
    for x, y, a in zip(id_v, iq_v, af_v):
        ax.annotate(f"{a:.2f}", (x, y), textcoords="offset points",
                    xytext=(5, 4), fontsize=7.5, color='black')

    # 방법 B 다항식 등고선
    pad = 80
    id_g = np.linspace(id_v.min() - pad, id_v.max() + pad, 50)
    iq_g = np.linspace(max(0, iq_v.min() - pad), iq_v.max() + pad, 50)
    ID, IQ = np.meshgrid(id_g, iq_g)
    AF_fit = af_from_poly3d(spd, ID.ravel(), IQ.ravel()).reshape(ID.shape)
    ct = ax.contour(ID, IQ, AF_fit, levels=8, cmap='coolwarm', alpha=0.65, linewidths=0.9)
    ax.clabel(ct, fmt="%.2f", fontsize=7.5)

    plt.colorbar(sc, ax=ax, label="AF [-]", shrink=0.85)
    ax.set_xlabel("$I_d$ [A, peak]", fontsize=9)
    ax.set_ylabel("$I_q$ [A, peak]", fontsize=9)
    ax.set_title(f"{spd/1000:.0f} kRPM", fontsize=11, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)

# ── 전류별 AF vs Speed 곡선 (방법 A 피팅선 포함) ──────────────────────────
ax_s = axes[-1]
unique_currents_v = sorted(set(round(p["current_rms"], 1) for p in af_points))
n_uc = len(unique_currents_v)
colors_v = [plt.cm.viridis(i / max(1, n_uc - 1)) for i in range(n_uc)]

for k, curr in enumerate(unique_currents_v):
    pts_c = sorted(
        [p for p in af_points if np.isclose(p["current_rms"], curr, atol=0.5)],
        key=lambda x: x["speed_rpm"]
    )
    spds_c = [p["speed_rpm"] for p in pts_c]
    afs_c  = [p["AF"]        for p in pts_c]
    ax_s.plot(spds_c, afs_c, marker='o', linestyle='-',
              color=colors_v[k], linewidth=1.6,
              label=f"$I_{{rms}}$={curr:.0f} A")

# 방법 A 피팅선 (최대 전류 기준)
spd_line  = np.linspace(min(unique_speeds_v) * 0.8, max(unique_speeds_v) * 1.05, 300)
af_A_line = af_from_speed_only(spd_line)
ax_s.plot(spd_line, af_A_line, 'k--', linewidth=2.0,
          label=f"Poly-A fit ($I_{{rms}}$={max_curr:.0f} A)")
ax_s.axhline(y=1.0, color='green', linestyle=':', linewidth=1.5, alpha=0.8, label="AF = 1")

ax_s.set_xlabel("Speed [RPM]", fontsize=9)
ax_s.set_ylabel("AF = FullFEA / Hybrid  [-]", fontsize=9)
ax_s.set_title("AF vs Speed (전류별)", fontsize=11, fontweight='bold')
ax_s.legend(fontsize=8, loc='upper left', framealpha=0.85)
ax_s.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("map_exports/AF_map_visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: map_exports/AF_map_visualization.png")

# [8] 보정 적용 검증 + Motor-CAD Lab 수식 최종 출력

- **방법 A**: Motor-CAD Lab → `Calculation → Custom Losses`에 직접 붙여 넣을 수식 출력
- **방법 B**: Python에서 `af_from_poly3d()`를 이용한 포인트별 보정 검증
- 결과를 `map_exports/AF_polynomial_model.json`에 저장

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# [8] 보정 적용 검증 + Motor-CAD Lab 수식 최종 출력
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import json
from pathlib import Path

# ── 방법 B: 3D 다항식 포인트별 검증 ─────────────────────────────────────────
print("=== 방법 B 검증: af_from_poly3d() 보정 정확도 ===\n")
print(f"{'Spd[kRPM]':>10} {'Id':>7} {'Iq':>7} {'H_AC':>8} "
      f"{'F_AC':>8} {'AF_ref':>7} {'AF_est':>7} {'Corr_AC':>9} {'Err%':>7}")
print("─" * 78)

err_pcts = []
for p in af_points:
    spd   = p["speed_rpm"]
    id_a  = p["id_A"]
    iq_a  = p["iq_A"]
    h_ac  = p["hybrid_ac_kW"]
    f_ac  = p["fea_ac_kW"]
    af_ref = p["AF"]

    af_est  = float(af_from_poly3d(spd, id_a, iq_a))
    corr_ac = h_ac * af_est
    err_pct = (corr_ac - f_ac) / (f_ac + 1e-9) * 100
    err_pcts.append(err_pct)

    print(f"{spd/1000:>10.1f} {id_a:>7.1f} {iq_a:>7.1f} {h_ac:>8.3f} "
          f"{f_ac:>8.3f} {af_ref:>7.3f} {af_est:>7.3f} {corr_ac:>9.3f} {err_pct:>7.1f}%")

err_arr = np.array(err_pcts)
print(f"\n  보정 오차 통계:  MAE={np.abs(err_arr).mean():.2f}%,  "
      f"MaxAE={np.abs(err_arr).max():.2f}%,  std={err_arr.std():.2f}%")

# ── 방법 A: Motor-CAD Lab 수식 최종 출력 ─────────────────────────────────────
a2, a1, a0 = _coeff_A
lab_formula_extra = (
    f"(({a2:.6f}*(Speed/1000)^2 "
    f"+ {a1:.6f}*(Speed/1000) "
    f"+ {a0:.6f}) - 1) * Stator_Copper_Loss_AC"
)

print("\n" + "=" * 70)
print("=== Motor-CAD Lab 커스텀 손실 수식 (방법 A) ===")
print("=" * 70)
print("\n  [적용 방법]")
print("  1. Lab 모듈 → Calculation → Custom Losses → Add Custom Loss")
print("  2. Name  : AC_loss_correction")
print("  3. Type  : Electrical")
print("  4. Function:")
print(f"\n     {lab_formula_extra}")
print("\n  [해석]")
print("  · 위 수식 = (AF - 1) × Hybrid_AC  = 보정 추가분 (Extra 손실)")
print("  · Motor-CAD는 이 값을 Hybrid AC 손실에 더해 최종 AC 손실을 계산")
print(f"  · 최대 전류 I_rms = {max_curr:.0f} A 기준 피팅 → 저전류에서는 보수적")
print("  · 전운전영역 정확도가 필요하면 방법 B (Python af_from_poly3d) 사용")

# ── 결과 저장 ─────────────────────────────────────────────────────────────────
export_data = {
    "method_A_poly2_speed_only": {
        "basis_current_Arms": float(max_curr),
        "coeffs_a2_a1_a0":    _coeff_A.tolist(),
        "formula_speed_kRPM": f"AF(s) = {a2:.6f}*s^2 + {a1:.6f}*s + {a0:.6f}",
        "lab_custom_loss_formula": lab_formula_extra,
    },
    "method_B_poly2_3D": {
        "feature_labels": _feat_labels,
        "coefficients":   _coeff_B.tolist(),
        "rmse":           float(rmse_B),
        "R2":             float(r2_B),
        "usage": "corrected_kW = af_from_poly3d(speed_rpm, id_peak_A, iq_peak_A) * hybrid_ac_kW",
    },
    "af_points": af_points,
}

Path("map_exports").mkdir(exist_ok=True)
save_path = Path("map_exports/AF_polynomial_model.json")
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)
print(f"\n  ✓ 저장 완료: {save_path}")

# ── 사용 예시 ─────────────────────────────────────────────────────────────────
print("\n=== Python 보정 함수 사용 예시 ===")
examples = [
    (2000,   0.0, 460.0 * np.sqrt(2), 1.0),
    (8000,  -150, 500.0, 2.5),
    (15000, -300, 400.0, 4.2),
]
print(f"{'Speed[RPM]':>12} {'Id[A]':>8} {'Iq[A]':>8} "
      f"{'H_AC[kW]':>10} {'AF_est':>8} {'Corr_AC[kW]':>13}")
print("─" * 65)
for spd, id_e, iq_e, h_e in examples:
    af_e  = float(af_from_poly3d(spd, id_e, iq_e))
    ca_e  = h_e * af_e
    print(f"{spd:>12} {id_e:>8.1f} {iq_e:>8.1f} {h_e:>10.2f} {af_e:>8.3f} {ca_e:>13.3f}")